# Automated Data Cleaning & Machine Learning System

A comprehensive, GUI-driven tool that:
1. Loads CSV / XLSX / XLS files via a Tkinter file dialog
2. Profiles the dataset (structure, missing values, outliers, data quality)
3. Cleans the data automatically with a full audit log
4. Trains baseline ML models (classification / regression / clustering)
5. Exports the cleaned dataset and a cleaning report

**Run every cell top-to-bottom.** The GUI buttons drive the workflow.

---
## 1 · Imports

In [ ]:
import warnings, os, sys, re, json, datetime, traceback, io
from pathlib import Path
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('TkAgg')          # must come before pyplot import for Tkinter GUI
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, OrdinalEncoder, LabelEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    RandomForestRegressor, GradientBoostingRegressor,
)
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, ConfusionMatrixDisplay,
    mean_absolute_error, mean_squared_error, r2_score,
    silhouette_score,
)
import joblib

import tkinter as tk
from tkinter import ttk, filedialog, messagebox, scrolledtext

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print('All imports OK  ·  Python', sys.version.split()[0])

---
## 2 · Configuration

In [ ]:
# ── Project paths ──────────────────────────────────────────────────
PROJECT_DIR  = Path.cwd()
DATA_DIR     = PROJECT_DIR / 'data'
OUTPUT_DIR   = PROJECT_DIR / 'output'
REPORTS_DIR  = PROJECT_DIR / 'reports'
MODELS_DIR   = PROJECT_DIR / 'models'

for d in (DATA_DIR, OUTPUT_DIR, REPORTS_DIR, MODELS_DIR):
    d.mkdir(exist_ok=True)

# ── Cleaning thresholds ────────────────────────────────────────────
MISSING_THRESHOLD_DROP_COL = 0.70    # drop column if >70% missing
HIGH_CARDINALITY_THRESHOLD = 50      # warn if >50 unique values
OUTLIER_IQR_FACTOR = 1.5
OUTLIER_ZSCORE_THRESHOLD = 3.0
NEAR_CONSTANT_THRESHOLD = 0.98       # >98% same value

# ── Common missing-value representations ──────────────────────────
MISSING_REPS = {
    '', ' ', 'n/a', 'na', 'nan', 'null', 'none',
    'unknown', '?', '-', '--', 'missing', 'n.a.', 'n.a',
    'not available', 'not applicable', 'undefined',
}

# ── ML defaults ───────────────────────────────────────────────────
TEST_SIZE = 0.20
RANDOM_STATE = 42
CV_FOLDS = 5

print('Configuration loaded.')

---
## 3 · Data Loading Utilities

In [ ]:
def load_dataset(filepath: str) -> pd.DataFrame:
    """Load CSV / XLSX / XLS into a DataFrame with error handling."""
    p = Path(filepath)
    if not p.exists():
        raise FileNotFoundError(f'File not found: {p}')
    ext = p.suffix.lower()
    try:
        if ext == '.csv':
            for enc in ('utf-8', 'latin-1', 'cp1252'):
                try:
                    df = pd.read_csv(p, encoding=enc, low_memory=False)
                    break
                except UnicodeDecodeError:
                    continue
            else:
                raise ValueError('Unable to decode CSV. Tried utf-8 / latin-1 / cp1252.')
        elif ext == '.xlsx':
            df = pd.read_excel(p, engine='openpyxl')
        elif ext == '.xls':
            df = pd.read_excel(p, engine='xlrd')
        else:
            raise ValueError(f'Unsupported format: {ext}')
    except Exception as e:
        raise RuntimeError(f'Error loading file: {e}')

    if df.empty:
        raise ValueError('The file is empty or contains no data.')
    if df.shape[1] < 1:
        raise ValueError('The dataset has no columns.')
    return df


print('Data loading utilities ready.')

---
## 4 · Dataset Profiling

In [ ]:
class DatasetProfiler:
    """Generate a comprehensive data-quality report."""

    def __init__(self, df: pd.DataFrame):
        self.df = df
        self.report = {}

    def _profile_structure(self):
        df = self.df
        buf = io.StringIO()
        df.info(buf=buf)
        self.report['structure'] = {
            'rows': len(df),
            'columns': len(df.columns),
            'column_names': list(df.columns),
            'dtypes': df.dtypes.astype(str).to_dict(),
            'memory_mb': round(df.memory_usage(deep=True).sum() / 1e6, 2),
            'duplicate_rows': int(df.duplicated().sum()),
            'info': buf.getvalue(),
        }

    def _profile_missing(self):
        df = self.df
        missing = df.isnull().sum()
        pct = (missing / len(df) * 100).round(2)
        self.report['missing'] = pd.DataFrame({
            'missing_count': missing,
            'missing_pct': pct,
        }).sort_values('missing_pct', ascending=False)
        self.report['excessive_missing_cols'] = list(
            pct[pct > MISSING_THRESHOLD_DROP_COL * 100].index
        )

    def _profile_numeric(self):
        num_cols = self.df.select_dtypes(include='number').columns.tolist()
        if not num_cols:
            self.report['numeric'] = None
            return
        desc = self.df[num_cols].describe().T
        skew = self.df[num_cols].skew()
        desc['skewness'] = skew
        q1 = self.df[num_cols].quantile(0.25)
        q3 = self.df[num_cols].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - OUTLIER_IQR_FACTOR * iqr
        upper = q3 + OUTLIER_IQR_FACTOR * iqr
        outlier_counts = ((self.df[num_cols] < lower) | (self.df[num_cols] > upper)).sum()
        desc['outlier_count'] = outlier_counts
        self.report['numeric'] = desc

    def _profile_categorical(self):
        cat_cols = self.df.select_dtypes(include=['object', 'category']).columns.tolist()
        if not cat_cols:
            self.report['categorical'] = None
            return
        info = {}
        for c in cat_cols:
            vc = self.df[c].value_counts(dropna=False)
            n_unique = self.df[c].nunique(dropna=True)
            info[c] = {
                'n_unique': n_unique,
                'top_values': vc.head(5).to_dict(),
                'rare_categories': int((vc < max(2, len(self.df) * 0.01)).sum()),
                'missing': int(self.df[c].isnull().sum()),
                'high_cardinality': n_unique > HIGH_CARDINALITY_THRESHOLD,
            }
        self.report['categorical'] = info

    def _profile_datetime(self):
        date_cols = []
        for c in self.df.select_dtypes(include='object').columns:
            sample = self.df[c].dropna().head(50)
            if sample.empty:
                continue
            try:
                pd.to_datetime(sample, infer_datetime_format=True)
                date_cols.append(c)
            except Exception:
                pass
        self.report['detected_date_cols'] = date_cols

    def _profile_quality(self):
        issues = []
        df = self.df

        n_dup = int(df.duplicated().sum())
        if n_dup:
            issues.append(f'{n_dup} duplicate row(s) found.')

        for c in df.columns:
            col = df[c]
            if col.dropna().nunique() <= 1:
                issues.append(f'Column "{c}" is constant (single unique value).')
            elif col.dropna().nunique() > 1:
                mode_frac = col.value_counts(normalize=True, dropna=True).iloc[0]
                if mode_frac >= NEAR_CONSTANT_THRESHOLD:
                    issues.append(f'Column "{c}" is near-constant ({mode_frac:.1%} same value).')

            if col.dtype == 'object':
                bad = col.dropna().apply(lambda x: str(x).strip().lower() in MISSING_REPS)
                n_bad = int(bad.sum())
                if n_bad:
                    issues.append(f'Column "{c}" has {n_bad} disguised missing value(s).')
                n_ws = int((col.dropna() != col.dropna().str.strip()).sum())
                if n_ws:
                    issues.append(f'Column "{c}" has {n_ws} value(s) with leading/trailing whitespace.')
                numeric_frac = col.dropna().apply(lambda x: str(x).replace('.', '', 1).replace('-', '', 1).isdigit()).mean()
                if 0.2 < numeric_frac < 0.8:
                    issues.append(f'Column "{c}" has mixed numeric/string values ({numeric_frac:.0%} numeric).')

            if col.dropna().nunique() == len(col.dropna()) and len(col.dropna()) > 10:
                if col.dtype == 'object' or pd.api.types.is_integer_dtype(col):
                    issues.append(f'Column "{c}" looks like an ID column (all unique values).')

        self.report['quality_issues'] = issues

    def profile(self):
        self._profile_structure()
        self._profile_missing()
        self._profile_numeric()
        self._profile_categorical()
        self._profile_datetime()
        self._profile_quality()
        return self.report

    def summary_text(self):
        r = self.report
        lines = []
        s = r['structure']
        lines.append(f"Rows: {s['rows']}  |  Columns: {s['columns']}  |  Memory: {s['memory_mb']} MB")
        lines.append(f"Duplicate rows: {s['duplicate_rows']}")
        lines.append('')
        m = r['missing']
        cols_with_missing = m[m['missing_count'] > 0]
        if not cols_with_missing.empty:
            lines.append('Columns with missing values:')
            for idx, row in cols_with_missing.iterrows():
                lines.append(f"  {idx}: {int(row['missing_count'])} ({row['missing_pct']:.1f}%)")
        else:
            lines.append('No missing values detected.')
        lines.append('')
        qi = r['quality_issues']
        if qi:
            lines.append(f'{len(qi)} data-quality issue(s):')
            for issue in qi:
                lines.append(f'  ⚠ {issue}')
        else:
            lines.append('No data-quality issues detected.')
        lines.append('')
        if r['detected_date_cols']:
            lines.append(f"Detected date columns: {r['detected_date_cols']}")
        return '\n'.join(lines)


print('DatasetProfiler ready.')

---
## 5 · Automated Cleaning Engine

In [ ]:
class CleaningAuditLog:
    def __init__(self):
        self.entries = []

    def add(self, column, operation, affected, detail='', automatic=True):
        self.entries.append({
            'column': column,
            'operation': operation,
            'affected_rows': affected,
            'detail': detail,
            'automatic': automatic,
            'timestamp': datetime.datetime.now().isoformat(timespec='seconds'),
        })

    def to_dataframe(self):
        if not self.entries:
            return pd.DataFrame(columns=['column', 'operation', 'affected_rows', 'detail', 'automatic', 'timestamp'])
        return pd.DataFrame(self.entries)

    def summary_text(self):
        if not self.entries:
            return 'No cleaning operations performed.'
        lines = [f'Cleaning Audit Log  ({len(self.entries)} operations)', '-' * 60]
        for e in self.entries:
            auto = 'AUTO' if e['automatic'] else 'MANUAL'
            lines.append(
                f"[{auto}] {e['column']:25s} | {e['operation']:30s} | "
                f"rows={e['affected_rows']}  | {e['detail']}"
            )
        return '\n'.join(lines)

class AutoCleaner:
    def __init__(self, df, profile_report):
        self.original = df
        self.df = df.copy()
        self.report = profile_report
        self.log = CleaningAuditLog()

    def clean_text(self):
        obj_cols = self.df.select_dtypes(include='object').columns
        for c in obj_cols:
            col = self.df[c]
            stripped = col.str.strip()
            n_ws = int((col.fillna('') != stripped.fillna('')).sum())
            if n_ws:
                self.df[c] = stripped
                self.log.add(c, 'strip whitespace', n_ws)
            mask = self.df[c].apply(
                lambda x: isinstance(x, str) and x.strip().lower() in MISSING_REPS
            )
            n_disguised = int(mask.sum())
            if n_disguised:
                self.df.loc[mask, c] = np.nan
                self.log.add(c, 'replace disguised missing→NaN', n_disguised,
                             detail='values like N/A, null, ?, unknown')

    def convert_dates(self):
        for c in self.report.get('detected_date_cols', []):
            if c not in self.df.columns:
                continue
            try:
                converted = pd.to_datetime(self.df[c], infer_datetime_format=True, errors='coerce')
                n_ok = int(converted.notna().sum())
                n_orig_notna = int(self.df[c].notna().sum())
                if n_ok >= n_orig_notna * 0.8:
                    self.df[c] = converted
                    self.log.add(c, 'convert to datetime', n_ok,
                                 f'{n_ok}/{n_orig_notna} values converted')
            except Exception:
                pass

    def convert_types(self):
        for c in self.df.select_dtypes(include='object').columns:
            col = self.df[c].dropna()
            if col.empty:
                continue
            numeric = pd.to_numeric(col, errors='coerce')
            if numeric.notna().sum() >= len(col) * 0.9:
                self.df[c] = pd.to_numeric(self.df[c], errors='coerce')
                self.log.add(c, 'convert to numeric', int(numeric.notna().sum()), 'object column was mostly numeric')
                continue
            lower_vals = set(col.astype(str).str.strip().str.lower().unique())
            bool_maps = [{'true', 'false'}, {'yes', 'no'}, {'1', '0'}, {'t', 'f'}, {'y', 'n'}]
            for bm in bool_maps:
                if lower_vals <= bm | {np.nan}:
                    true_set = {'true', 'yes', '1', 't', 'y'}
                    self.df[c] = self.df[c].apply(
                        lambda x: True if isinstance(x, str) and x.strip().lower() in true_set else (False if isinstance(x, str) else x)
                    )
                    self.log.add(c, 'convert to boolean', len(col), f'values were {lower_vals}')
                    break

    def remove_duplicates(self):
        n_dup = int(self.df.duplicated().sum())
        if n_dup:
            self.df = self.df.drop_duplicates().reset_index(drop=True)
            self.log.add('(all)', 'remove duplicate rows', n_dup,
                         f'{n_dup} exact duplicate rows removed')

    def drop_excessive_missing(self):
        for c in list(self.report.get('excessive_missing_cols', [])):
            if c in self.df.columns:
                pct = self.df[c].isnull().mean() * 100
                self.df = self.df.drop(columns=[c])
                self.log.add(c, 'drop column (excessive missing)', 0,
                             f'{pct:.1f}% missing – above {MISSING_THRESHOLD_DROP_COL*100:.0f}% threshold')

    def impute_missing(self):
        for c in self.df.columns:
            n_miss = int(self.df[c].isnull().sum())
            if n_miss == 0:
                continue
            if pd.api.types.is_numeric_dtype(self.df[c]):
                median_val = self.df[c].median()
                self.df[c] = self.df[c].fillna(median_val)
                self.log.add(c, 'impute missing (median)', n_miss,
                             f'filled with median={median_val:.4g}')
            elif pd.api.types.is_datetime64_any_dtype(self.df[c]):
                self.log.add(c, 'missing datetime kept as NaT', n_miss,
                             'no imputation – inventing dates is unsafe')
            elif self.df[c].dtype == 'object' or self.df[c].dtype.name == 'category':
                mode = self.df[c].mode()
                if not mode.empty and n_miss <= len(self.df) * 0.3:
                    fill = mode.iloc[0]
                    self.df[c] = self.df[c].fillna(fill)
                    self.log.add(c, 'impute missing (mode)', n_miss, f'filled with mode="{fill}"')
                else:
                    self.df[c] = self.df[c].fillna('Unknown')
                    self.log.add(c, 'impute missing ("Unknown")', n_miss, 'too many missing for mode imputation')
            elif pd.api.types.is_bool_dtype(self.df[c]):
                mode = self.df[c].mode()
                fill = mode.iloc[0] if not mode.empty else False
                self.df[c] = self.df[c].fillna(fill)
                self.log.add(c, 'impute missing (mode)', n_miss, f'filled with {fill}')

    def handle_outliers(self):
        num_cols = self.df.select_dtypes(include='number').columns
        for c in num_cols:
            col = self.df[c].dropna()
            if col.empty or col.nunique() <= 2:
                continue
            q1, q3 = col.quantile(0.25), col.quantile(0.75)
            iqr = q3 - q1
            if iqr == 0:
                continue
            lower = q1 - OUTLIER_IQR_FACTOR * iqr
            upper = q3 + OUTLIER_IQR_FACTOR * iqr
            outliers = (self.df[c] < lower) | (self.df[c] > upper)
            n_out = int(outliers.sum())
            if n_out == 0:
                continue
            pct_outlier = n_out / len(self.df)
            if pct_outlier <= 0.05:
                self.df[c] = self.df[c].clip(lower=lower, upper=upper)
                self.log.add(c, 'cap outliers (winsorize)', n_out,
                             f'IQR method: capped to [{lower:.4g}, {upper:.4g}]')
            else:
                self.log.add(c, 'outliers retained', n_out,
                             f'{pct_outlier:.1%} are outliers – too many to cap safely')

    def clean(self):
        self.clean_text()
        self.convert_dates()
        self.convert_types()
        self.remove_duplicates()
        self.drop_excessive_missing()
        self.impute_missing()
        self.handle_outliers()
        return self.df

print('AutoCleaner ready.')

---
## 6 · Visualization Helpers

In [ ]:
def plot_missing_values(df, title='Missing Values'):
    missing = df.isnull().sum()
    missing = missing[missing > 0].sort_values(ascending=False)
    if missing.empty:
        print('No missing values to plot.')
        return
    fig, ax = plt.subplots(figsize=(10, max(4, len(missing) * 0.4)))
    missing.plot.barh(ax=ax, color='salmon')
    ax.set_title(title)
    ax.set_xlabel('Missing Count')
    plt.tight_layout()
    plt.show()

def plot_numeric_distributions(df, max_cols=12):
    num_cols = df.select_dtypes(include='number').columns[:max_cols]
    if len(num_cols) == 0:
        print('No numeric columns to plot.')
        return
    n = len(num_cols)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    for i, c in enumerate(num_cols):
        sns.histplot(df[c].dropna(), kde=True, ax=axes_flat[i], color='steelblue')
        axes_flat[i].set_title(c)
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.suptitle('Numeric Distributions', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_boxplots(df, max_cols=12):
    num_cols = df.select_dtypes(include='number').columns[:max_cols]
    if len(num_cols) == 0:
        return
    n = len(num_cols)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
    axes_flat = np.array(axes).flatten() if n > 1 else [axes]
    for i, c in enumerate(num_cols):
        sns.boxplot(x=df[c].dropna(), ax=axes_flat[i], color='lightblue')
        axes_flat[i].set_title(c)
    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)
    plt.suptitle('Box Plots (Outlier View)', y=1.02, fontsize=14)
    plt.tight_layout()
    plt.show()

def plot_correlation_matrix(df):
    num_df = df.select_dtypes(include='number')
    if num_df.shape[1] < 2:
        print('Need ≥2 numeric columns for correlation matrix.')
        return
    corr = num_df.corr()
    fig, ax = plt.subplots(figsize=(min(14, num_df.shape[1] + 2), min(12, num_df.shape[1] + 1)))
    sns.heatmap(corr, annot=corr.shape[0] <= 15, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
    ax.set_title('Correlation Matrix')
    plt.tight_layout()
    plt.show()

def plot_categorical_frequencies(df, max_cols=6):
    cat_cols = df.select_dtypes(include=['object', 'category']).columns[:max_cols]
    if len(cat_cols) == 0:
        return
    for c in cat_cols:
        vc = df[c].value_counts().head(15)
        fig, ax = plt.subplots(figsize=(8, max(3, len(vc) * 0.35)))
        vc.plot.barh(ax=ax, color='mediumpurple')
        ax.set_title(f'{c} – Top Categories')
        ax.set_xlabel('Count')
        plt.tight_layout()
        plt.show()

print('Visualization helpers ready.')

---
## 7 · ML Engine

In [ ]:
class MLEngine:
    def __init__(self, df_cleaned, target_col=None):
        self.df = df_cleaned.copy()
        self.target_col = target_col
        self.task = None
        self.results = {}
        self.best_model = None
        self.best_name = None
        self.X_train = self.X_test = self.y_train = self.y_test = None
        self.feature_names = []

    def detect_task(self):
        if self.target_col is None:
            self.task = 'clustering'
            return self.task
        y = self.df[self.target_col]
        if pd.api.types.is_numeric_dtype(y) and y.nunique() > 20:
            self.task = 'regression'
        else:
            self.task = 'classification'
        return self.task

    def _prepare(self):
        drop_cols = []
        for c in self.df.columns:
            if pd.api.types.is_datetime64_any_dtype(self.df[c]):
                drop_cols.append(c)
        if self.target_col:
            drop_cols.append(self.target_col)

        feature_cols = [c for c in self.df.columns if c not in drop_cols]
        X = self.df[feature_cols].copy()

        num_cols = X.select_dtypes(include='number').columns.tolist()
        cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
        bool_cols = X.select_dtypes(include='bool').columns.tolist()

        cat_cols = [c for c in cat_cols if X[c].nunique() <= HIGH_CARDINALITY_THRESHOLD]

        for bc in bool_cols:
            X[bc] = X[bc].astype(int)
        num_cols = num_cols + bool_cols

        self.feature_names = num_cols + cat_cols

        numeric_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', RobustScaler()),
        ])
        categorical_pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ])

        transformers = []
        if num_cols:
            transformers.append(('num', numeric_pipe, num_cols))
        if cat_cols:
            transformers.append(('cat', categorical_pipe, cat_cols))

        if not transformers:
            raise ValueError('No usable features for ML after preprocessing.')

        self.preprocessor = ColumnTransformer(transformers, remainder='drop')
        
        if len(self.feature_names) == 0:
            raise ValueError('No usable features after filtering.')
            
        return X[self.feature_names].copy()

    def run_classification(self):
        X = self._prepare()
        y = self.df[self.target_col]

        if y.dtype == 'object' or y.dtype.name == 'category':
            self._le = LabelEncoder()
            y = pd.Series(self._le.fit_transform(y), name=self.target_col)
        else:
            self._le = None

        if y.nunique() < 2:
            raise ValueError('Target has only 1 class. Classification impossible.')
        if len(y) < 10:
            raise ValueError('Dataset too small for ML (need ≥10 samples).')

        try:
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
            )
        except ValueError:
            self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
                X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
            )

        models = {
            'LogisticRegression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
            'RandomForest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE),
            'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
        }

        is_binary = y.nunique() == 2

        for name, model in models.items():
            pipe = Pipeline([
                ('preprocessor', self.preprocessor),
                ('model', model),
            ])
            pipe.fit(self.X_train, self.y_train)
            y_pred = pipe.predict(self.X_test)
            avg = 'binary' if is_binary else 'weighted'
            metrics = {
                'accuracy': round(accuracy_score(self.y_test, y_pred), 4),
                'precision': round(precision_score(self.y_test, y_pred, average=avg, zero_division=0), 4),
                'recall': round(recall_score(self.y_test, y_pred, average=avg, zero_division=0), 4),
                'f1': round(f1_score(self.y_test, y_pred, average=avg, zero_division=0), 4),
            }
            try:
                if is_binary:
                    y_proba = pipe.predict_proba(self.X_test)[:, 1]
                    metrics['roc_auc'] = round(roc_auc_score(self.y_test, y_proba), 4)
                else:
                    y_proba = pipe.predict_proba(self.X_test)
                    metrics['roc_auc'] = round(roc_auc_score(self.y_test, y_proba, multi_class='ovr', average='weighted'), 4)
            except Exception:
                metrics['roc_auc'] = None
            metrics['confusion_matrix'] = confusion_matrix(self.y_test, y_pred)
            metrics['pipeline'] = pipe
            self.results[name] = metrics

        self.best_name = max(self.results, key=lambda k: self.results[k]['f1'])
        self.best_model = self.results[self.best_name]['pipeline']

    def run_regression(self):
        X = self._prepare()
        y = self.df[self.target_col].astype(float)

        if len(y) < 10:
            raise ValueError('Dataset too small for ML (need ≥10 samples).')

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
        )

        models = {
            'LinearRegression': LinearRegression(),
            'RandomForest': RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE),
            'GradientBoosting': GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE),
        }

        for name, model in models.items():
            pipe = Pipeline([
                ('preprocessor', self.preprocessor),
                ('model', model),
            ])
            pipe.fit(self.X_train, self.y_train)
            y_pred = pipe.predict(self.X_test)
            metrics = {
                'MAE': round(mean_absolute_error(self.y_test, y_pred), 4),
                'MSE': round(mean_squared_error(self.y_test, y_pred), 4),
                'RMSE': round(np.sqrt(mean_squared_error(self.y_test, y_pred)), 4),
                'R2': round(r2_score(self.y_test, y_pred), 4),
                'pipeline': pipe,
                'y_pred': y_pred,
            }
            self.results[name] = metrics

        self.best_name = max(self.results, key=lambda k: self.results[k]['R2'])
        self.best_model = self.results[self.best_name]['pipeline']

    def run_clustering(self, n_clusters=3):
        X = self._prepare()
        if len(X) < n_clusters:
            raise ValueError(f'Need at least {n_clusters} samples for {n_clusters} clusters.')
        pipe = Pipeline([
            ('preprocessor', self.preprocessor),
        ])
        X_transformed = pipe.fit_transform(X)
        kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_STATE, n_init=10)
        labels = kmeans.fit_predict(X_transformed)
        sil = None
        if len(set(labels)) > 1:
            sil = round(silhouette_score(X_transformed, labels), 4)
        cluster_sizes = pd.Series(labels).value_counts().sort_index().to_dict()
        self.results['KMeans'] = {
            'n_clusters': n_clusters,
            'cluster_sizes': cluster_sizes,
            'silhouette_score': sil,
            'labels': labels,
            'X_transformed': X_transformed,
        }
        self.best_name = 'KMeans'

    def run(self):
        self.detect_task()
        if self.task == 'classification':
            self.run_classification()
        elif self.task == 'regression':
            self.run_regression()
        else:
            self.run_clustering()
        return self.results

    def summary_text(self):
        lines = [f'ML Task: {self.task}', '']
        if self.target_col:
            lines.append(f'Target column: {self.target_col}')
            lines.append(f'Features used: {len(self.feature_names)}')
            if self.X_train is not None:
                lines.append(f'Train size: {len(self.X_train)}  |  Test size: {len(self.X_test)}')
        lines.append(f'Evaluation: {TEST_SIZE*100:.0f}% holdout test set')
        lines.append('')
        if self.task == 'classification':
            header = f'{"Model":25s} {"Accuracy":>10s} {"Precision":>10s} {"Recall":>10s} {"F1":>10s} {"ROC-AUC":>10s}'
            lines.append(header)
            lines.append('-' * len(header))
            for name, m in self.results.items():
                roc = f"{m['roc_auc']:.4f}" if m.get('roc_auc') is not None else 'N/A'
                lines.append(
                    f"{name:25s} {m['accuracy']:10.4f} {m['precision']:10.4f} "
                    f"{m['recall']:10.4f} {m['f1']:10.4f} {roc:>10s}"
                )
        elif self.task == 'regression':
            header = f'{"Model":25s} {"MAE":>10s} {"MSE":>12s} {"RMSE":>10s} {"R²":>10s}'
            lines.append(header)
            lines.append('-' * len(header))
            for name, m in self.results.items():
                lines.append(
                    f"{name:25s} {m['MAE']:10.4f} {m['MSE']:12.4f} "
                    f"{m['RMSE']:10.4f} {m['R2']:10.4f}"
                )
        elif self.task == 'clustering':
            m = self.results.get('KMeans', {})
            lines.append(f"Clusters: {m.get('n_clusters')}")
            lines.append(f"Cluster sizes: {m.get('cluster_sizes')}")
            sil = m.get('silhouette_score')
            lines.append(f"Silhouette Score: {sil if sil is not None else 'N/A'}")
        if self.best_name:
            lines.append(f'\nBest model: {self.best_name}')
        return '\n'.join(lines)

    def plot_results(self):
        if self.task == 'classification':
            self._plot_classification()
        elif self.task == 'regression':
            self._plot_regression()
        elif self.task == 'clustering':
            self._plot_clustering()

    def _plot_classification(self):
        names = list(self.results.keys())
        metric_keys = ['accuracy', 'precision', 'recall', 'f1']
        data = {k: [self.results[n][k] for n in names] for k in metric_keys}
        comp_df = pd.DataFrame(data, index=names)
        ax = comp_df.plot.bar(figsize=(10, 5), rot=0)
        ax.set_title('Classification Model Comparison')
        ax.set_ylim(0, 1.05)
        ax.legend(loc='lower right')
        plt.tight_layout()
        plt.show()
        cm = self.results[self.best_name]['confusion_matrix']
        fig, ax = plt.subplots(figsize=(6, 5))
        ConfusionMatrixDisplay(cm).plot(ax=ax, cmap='Blues')
        ax.set_title(f'Confusion Matrix – {self.best_name}')
        plt.tight_layout()
        plt.show()
        if self.y_train is not None:
            fig, ax = plt.subplots(figsize=(7, 4))
            pd.Series(self.y_train).value_counts().sort_index().plot.bar(ax=ax, color='teal')
            ax.set_title('Training Set Class Distribution')
            ax.set_xlabel('Class')
            ax.set_ylabel('Count')
            plt.tight_layout()
            plt.show()

    def _plot_regression(self):
        names = list(self.results.keys())
        r2_vals = [self.results[n]['R2'] for n in names]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.barh(names, r2_vals, color='mediumseagreen')
        ax.set_xlabel('R² Score')
        ax.set_title('Regression Model Comparison (R²)')
        plt.tight_layout()
        plt.show()
        y_pred = self.results[self.best_name]['y_pred']
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(self.y_test, y_pred, alpha=0.5, edgecolor='k', linewidth=0.5)
        lims = [min(self.y_test.min(), y_pred.min()), max(self.y_test.max(), y_pred.max())]
        ax.plot(lims, lims, 'r--', linewidth=1)
        ax.set_xlabel('Actual')
        ax.set_ylabel('Predicted')
        ax.set_title(f'Predicted vs Actual – {self.best_name}')
        plt.tight_layout()
        plt.show()

    def _plot_clustering(self):
        m = self.results.get('KMeans', {})
        labels = m.get('labels')
        X_t = m.get('X_transformed')
        if labels is None or X_t is None:
            return
        if X_t.shape[1] >= 2:
            pca = PCA(n_components=2)
            X2 = pca.fit_transform(X_t)
        else:
            X2 = X_t
        fig, ax = plt.subplots(figsize=(8, 6))
        scatter = ax.scatter(X2[:, 0], X2[:, 1], c=labels, cmap='viridis', alpha=0.6, edgecolor='k', linewidth=0.3)
        ax.set_title('KMeans Clustering (PCA 2D Projection)')
        ax.set_xlabel('PC1')
        ax.set_ylabel('PC2')
        plt.colorbar(scatter, label='Cluster')
        plt.tight_layout()
        plt.show()

    def save_best_model(self, path=None):
        if self.best_model is None:
            return None
        if path is None:
            path = MODELS_DIR / f'best_model_{self.best_name}.joblib'
        joblib.dump(self.best_model, path)
        return path

print('MLEngine ready.')

---
## 8 · Report Generation

In [ ]:
def generate_report(filepath, original_df, cleaned_df, audit_log, ml_summary=None) -> str:
    lines = [
        '=' * 60,
        'DATA CLEANING & ML REPORT',
        '=' * 60,
        f'Generated: {datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        f'Source file: {filepath}',
        '',
        '── Original Dataset ──',
        f'  Rows: {len(original_df)}',
        f'  Columns: {len(original_df.columns)}',
        f'  Missing values: {int(original_df.isnull().sum().sum())}',
        f'  Duplicate rows: {int(original_df.duplicated().sum())}',
        '',
        '── Cleaned Dataset ──',
        f'  Rows: {len(cleaned_df)}',
        f'  Columns: {len(cleaned_df.columns)}',
        f'  Missing values: {int(cleaned_df.isnull().sum().sum())}',
        f'  Duplicate rows: {int(cleaned_df.duplicated().sum())}',
        '',
        '── Cleaning Operations ──',
    ]
    if audit_log and audit_log.entries:
        for e in audit_log.entries:
            auto = 'AUTO' if e['automatic'] else 'MANUAL'
            lines.append(f"  [{auto}] {e['column']} | {e['operation']} | rows={e['affected_rows']} | {e['detail']}")
    else:
        lines.append('  No cleaning operations performed.')
    if ml_summary:
        lines.append('')
        lines.append('── ML Results ──')
        lines.append(ml_summary)
    lines.append('')
    lines.append('=' * 60)
    return '\n'.join(lines)

print('Report generator ready.')

---
## 9 · GUI Application

In [ ]:
class DataCleaningApp:
    def __init__(self):
        self.root = tk.Tk()
        self.root.title('Automated Data Cleaning & ML')
        self.root.geometry('960x750')
        self.root.minsize(800, 600)
        self.filepath = None
        self.original_df = None
        self.cleaned_df = None
        self.profile_report = None
        self.audit_log = None
        self.ml_engine = None
        self.ml_summary = None
        self._build_ui()

    def _build_ui(self):
        frm_top = ttk.LabelFrame(self.root, text='Dataset Selection', padding=10)
        frm_top.pack(fill='x', padx=10, pady=(10, 5))
        self.btn_select = ttk.Button(frm_top, text='Select CSV/Excel File', command=self._select_file)
        self.btn_select.pack(side='left', padx=(0, 10))
        self.lbl_filepath = ttk.Label(frm_top, text='No file selected', foreground='gray')
        self.lbl_filepath.pack(side='left', fill='x', expand=True)
        frm_btns = ttk.Frame(self.root, padding=5)
        frm_btns.pack(fill='x', padx=10, pady=5)
        self.btn_load = ttk.Button(frm_btns, text='Load Dataset', command=self._load_dataset, state='disabled')
        self.btn_load.pack(side='left', padx=5)
        self.btn_clean = ttk.Button(frm_btns, text='Clean Dataset', command=self._clean_dataset, state='disabled')
        self.btn_clean.pack(side='left', padx=5)
        self.btn_ml = ttk.Button(frm_btns, text='Run ML Analysis', command=self._run_ml, state='disabled')
        self.btn_ml.pack(side='left', padx=5)
        self.btn_export = ttk.Button(frm_btns, text='Export Cleaned Data', command=self._export_data, state='disabled')
        self.btn_export.pack(side='left', padx=5)
        frm_target = ttk.Frame(self.root, padding=5)
        frm_target.pack(fill='x', padx=10, pady=2)
        ttk.Label(frm_target, text='ML Target Column:').pack(side='left')
        self.target_var = tk.StringVar(value='(none – clustering)')
        self.cmb_target = ttk.Combobox(frm_target, textvariable=self.target_var, state='disabled', width=35)
        self.cmb_target.pack(side='left', padx=10)
        self.progress = ttk.Progressbar(self.root, mode='indeterminate')
        self.progress.pack(fill='x', padx=10, pady=5)
        self.lbl_status = ttk.Label(self.root, text='Ready.', foreground='blue')
        self.lbl_status.pack(fill='x', padx=10)
        self.txt_output = scrolledtext.ScrolledText(self.root, wrap='word', height=25, font=('Consolas', 9))
        self.txt_output.pack(fill='both', expand=True, padx=10, pady=(5, 10))

    def _log(self, text: str):
        self.txt_output.insert('end', text + '\n')
        self.txt_output.see('end')
        self.root.update_idletasks()

    def _set_status(self, text: str, color: str = 'blue'):
        self.lbl_status.config(text=text, foreground=color)
        self.root.update_idletasks()

    def _start_progress(self):
        self.progress.start(15)
        self.root.update_idletasks()

    def _stop_progress(self):
        self.progress.stop()
        self.root.update_idletasks()

    def _select_file(self):
        filetypes = [('All Supported', '*.csv;*.xlsx;*.xls'), ('CSV Files', '*.csv'), ('Excel Files', '*.xlsx;*.xls')]
        path = filedialog.askopenfilename(title='Select Dataset', filetypes=filetypes)
        if path:
            self.filepath = path
            self.lbl_filepath.config(text=path, foreground='black')
            self.btn_load.config(state='normal')
            self._set_status(f'File selected: {Path(path).name}')
            self._log(f'\n>>> File selected: {path}')

    def _load_dataset(self):
        self._start_progress()
        self._set_status('Loading dataset…')
        try:
            self.original_df = load_dataset(self.filepath)
            self._log(f'\n✓ Loaded {len(self.original_df)} rows × {len(self.original_df.columns)} columns.')
            self._set_status('Profiling dataset…')
            profiler = DatasetProfiler(self.original_df)
            self.profile_report = profiler.profile()
            self._log('\n── Dataset Profile ──')
            self._log(profiler.summary_text())
            cols = ['(none – clustering)'] + list(self.original_df.columns)
            self.cmb_target.config(values=cols, state='readonly')
            self.target_var.set(cols[0])
            self.btn_clean.config(state='normal')
            self._set_status('Dataset loaded and profiled.', 'green')
        except Exception as e:
            self._log(f'\n✗ ERROR: {e}')
            self._set_status(f'Error: {e}', 'red')
        finally:
            self._stop_progress()

    def _clean_dataset(self):
        self._start_progress()
        self._set_status('Cleaning dataset…')
        try:
            cleaner = AutoCleaner(self.original_df, self.profile_report)
            self.cleaned_df = cleaner.clean()
            self.audit_log = cleaner.log
            self._log('\n── Cleaning Complete ──')
            self._log(f'Original: {len(self.original_df)} rows × {len(self.original_df.columns)} cols')
            self._log(f'Cleaned:  {len(self.cleaned_df)} rows × {len(self.cleaned_df.columns)} cols')
            self._log(f'Missing before: {int(self.original_df.isnull().sum().sum())}  →  after: {int(self.cleaned_df.isnull().sum().sum())}')
            self._log('')
            self._log(self.audit_log.summary_text())
            cols = ['(none – clustering)'] + list(self.cleaned_df.columns)
            self.cmb_target.config(values=cols, state='readonly')
            if self.target_var.get() not in cols:
                self.target_var.set(cols[0])
            self.btn_ml.config(state='normal')
            self.btn_export.config(state='normal')
            self._set_status('Dataset cleaned successfully.', 'green')
        except Exception as e:
            self._log(f'\n✗ ERROR during cleaning: {e}')
            self._set_status(f'Error: {e}', 'red')
        finally:
            self._stop_progress()

    def _run_ml(self):
        self._start_progress()
        self._set_status('Running ML analysis…')
        try:
            target = self.target_var.get()
            if target == '(none – clustering)' or target == '':
                target = None
            self.ml_engine = MLEngine(self.cleaned_df, target_col=target)
            self.ml_engine.run()
            self.ml_summary = self.ml_engine.summary_text()
            self._log('\n── ML Results ──')
            self._log(self.ml_summary)
            saved = self.ml_engine.save_best_model()
            if saved:
                self._log(f'\n✓ Best model saved to: {saved}')
            self.ml_engine.plot_results()
            self._set_status('ML analysis complete.', 'green')
        except Exception as e:
            self._log(f'\n✗ ML ERROR: {e}')
            self._set_status(f'ML Error: {e}', 'red')
            traceback.print_exc()
        finally:
            self._stop_progress()

    def _export_data(self):
        if self.cleaned_df is None:
            messagebox.showwarning('Export', 'No cleaned dataset available.')
            return
        filetypes = [('CSV Files', '*.csv'), ('Excel Files', '*.xlsx')]
        default_name = Path(self.filepath).stem + '_cleaned' if self.filepath else 'cleaned_data'
        save_path = filedialog.asksaveasfilename(title='Save Cleaned Dataset', defaultextension='.csv', initialfile=default_name, initialdir=str(OUTPUT_DIR), filetypes=filetypes)
        if not save_path:
            return
        try:
            p = Path(save_path)
            if p.suffix.lower() == '.xlsx':
                self.cleaned_df.to_excel(p, index=False, engine='openpyxl')
            else:
                self.cleaned_df.to_csv(p, index=False)
            self._log(f'\n✓ Cleaned dataset exported to: {p}')
            report_path = REPORTS_DIR / (p.stem + '_report.txt')
            report_text = generate_report(self.filepath, self.original_df, self.cleaned_df, self.audit_log, self.ml_summary)
            report_path.write_text(report_text, encoding='utf-8')
            self._log(f'✓ Cleaning report saved to: {report_path}')
            self._set_status('Export complete.', 'green')
        except Exception as e:
            self._log(f'\n✗ Export ERROR: {e}')
            self._set_status(f'Export Error: {e}', 'red')

    def run(self):
        self.root.mainloop()

print('DataCleaningApp ready.')

---
## 10 · Launch GUI

Run the cell below to open the application window.

**Workflow:**
1. Click **Select CSV/Excel File** → pick a file
2. Click **Load Dataset** → see the profile in the output area
3. Click **Clean Dataset** → see the audit log
4. (Optional) Choose a target column for supervised ML, or leave *(none – clustering)*
5. Click **Run ML Analysis** → see metrics & plots
6. Click **Export Cleaned Data** → save CSV/XLSX + a report

In [ ]:
app = DataCleaningApp()
app.run()

---
## 11 · Post-GUI: Exploratory Data Analysis

After using the GUI, call these cells to generate additional visualisation in-notebook.

The variables `app.original_df`, `app.cleaned_df`, `app.audit_log`, and `app.ml_engine` are available.

In [ ]:
if app.original_df is not None:
    print('── Missing Values (Original) ──')
    plot_missing_values(app.original_df, title='Missing Values (Original)')

In [ ]:
if app.cleaned_df is not None:
    print('── Numeric Distributions (Cleaned) ──')
    plot_numeric_distributions(app.cleaned_df)

In [ ]:
if app.cleaned_df is not None:
    print('── Box Plots (Cleaned) ──')
    plot_boxplots(app.cleaned_df)

In [ ]:
if app.cleaned_df is not None:
    print('── Correlation Matrix (Cleaned) ──')
    plot_correlation_matrix(app.cleaned_df)

In [ ]:
if app.cleaned_df is not None:
    print('── Categorical Frequencies (Cleaned) ──')
    plot_categorical_frequencies(app.cleaned_df)

---
## 12 · Audit Log (Display)

In [ ]:
if app.audit_log is not None:
    display(app.audit_log.to_dataframe())
else:
    print('No audit log yet – run the GUI cleaning step first.')

---
## 13 · ML Results (Display)

In [ ]:
if app.ml_engine is not None:
    print(app.ml_engine.summary_text())
else:
    print('No ML results yet – run the GUI ML step first.')

---
## 14 · Summary

This notebook provides:

* **GUI-driven workflow** — select, load, clean, ML, export without typing paths
* **Comprehensive profiling** — structure, missing values, outliers, data quality
* **Non-destructive cleaning** — original DataFrame is always preserved
* **Full audit log** — every operation is recorded with column, count, and reason
* **Leakage-free ML** — `Pipeline` + `ColumnTransformer` applied after train/test split
* **Classification / Regression / Clustering** — auto-detected from target type
* **Baseline model comparison** — with proper metrics and visualisations
* **Export** — cleaned data (CSV/XLSX) + cleaning report (TXT)